# 02 — Statistics: exploration and validation
Tests `roche_poc.stats`. Everything here is pandas: no NLP, no LLM.

Goal: check that each indicator behaves sensibly on the real data before it is shown in the dashboard.

In [ ]:
# Works after `pip install -e .`; the fallback below covers a fresh clone.
import sys, pathlib
_src = pathlib.Path.cwd().resolve().parent / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
import pandas as pd
pd.set_option("display.max_columns", 30, "display.width", 160)

In [ ]:
from roche_poc import config
from roche_poc.data.build_tables import load_table
from roche_poc.stats import kpis, persistence, profile, trends, watchlist
snapshots = load_table(config.SNAPSHOTS_PATH)
comments = load_table(config.COMMENTS_PATH)
profile.portfolio_overview(snapshots)['status_counts']

## 1. Why the first prototype's trend was wrong
Counting *unique comments* per first appearance measures new comments, not activity. Counting *critical elements per snapshot* measures the real situation.

In [ ]:
old = comments.groupby('first_seen').size().rename('new unique comments')
new = kpis.status_timeline(snapshots)[list(config.CRITICAL_STATUSES)].sum(axis=1).rename('critical elements')
pd.concat([old, new], axis=1).plot(subplots=True, figsize=(9, 5), marker='o');

## 2. Persistence and alerts

In [ ]:
streaks = persistence.critical_streaks(snapshots)
streaks.sort_values('current_streak', ascending=False).head(10)

In [ ]:
wl = watchlist.build_watchlist(snapshots)
print(len(wl), 'alerts')
wl.groupby(['rule', 'severity']).size()

## 3. Rankings

In [ ]:
watchlist.rank_entities(snapshots, 'vendor', wl).head(10)

In [ ]:
watchlist.rank_entities(snapshots, 'mrp_controller', wl).head(10)

## 4. One profile, three levels
Same keys whatever the level.

In [ ]:
top_vendor = watchlist.rank_entities(snapshots, 'vendor', wl).iloc[0][config.COL_VENDOR]
p = profile.build_profile(snapshots, comments, 'vendor', top_vendor)
list(p), p['current']['status_counts'], p['watchlist']['n_high']

In [ ]:
p['top_materials']

In [ ]:
# Trend labels need at least MIN_POINTS_FOR_TREND snapshots
p['history']['critical_share_trend']